# ResearchGPT: Milad Saeedi Research Portfolio RAG

## A Retrieval-Augmented Generation System for Research, CV, and Technical Projects

This project builds a Retrieval-Augmented Generation (RAG) assistant that allows users to ask questions about Milad Saeedi’s:

- PhD thesis
- Peer-reviewed publications
- Curriculum vitae
- GitHub repositories
- Jupyter notebooks
- README files
- Python projects
- HTML project reports
- Technical documentation

The system retrieves relevant information from these sources and generates grounded answers with citations.

The knowledge base focuses on topics related to:

- Geospatial Machine Learning
- Environmental Science
- Air Pollution
- Machine Learning
- Computer Vision
- Large Language Models
- Multimodal Learning
- Scientific AI Applications

The RAG pipeline includes:

1. Document ingestion
2. Text extraction and cleaning
3. Document filtering
4. Text chunking
5. Embedding generation
6. FAISS vector indexing
7. Semantic retrieval
8. Cross-encoder reranking
9. Prompt construction
10. Grounded answer generation
11. Source citation
12. Gradio chatbot interface

Example questions include:

- What are Milad Saeedi’s main research contributions?
- Which projects involve computer vision?
- What machine learning methods has he used?
- How was mobile air pollution monitoring applied?
- What experience does he have with LLM fine-tuning?
- Which GitHub projects use PyTorch?
- What are the main contributions of his PhD thesis?

**Author:** Milad Saeedi

## 1. Install dependencies

The project uses:

- PyMuPDF for PDF extraction
- Beautiful Soup for HTML parsing
- `nbformat` for Jupyter notebooks
- Sentence Transformers for embeddings and reranking
- FAISS for vector similarity search
- Ollama for local language-model inference
- Gradio for the public chatbot interface

In [ ]:
%pip install -q \
    pymupdf \
    nbformat \
    beautifulsoup4 \
    sentence-transformers \
    faiss-cpu \
    pandas \
    numpy \
    ollama \
    gradio\
    rank-bm25

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import re

import faiss
import fitz
import gradio as gr
import nbformat
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from ollama import chat
from sentence_transformers import CrossEncoder, SentenceTransformer

## 2. Project configuration

This section defines:

- The root folder containing the knowledge-base files
- Supported file types
- Ignored folders
- PDF document labels
- Files that require two-column PDF extraction
- Page ranges that should not enter the searchable index

In [ ]:
DATA_DIR = Path("../data")

GITHUB_DIR = Path(
    "/Users/miladsaeedi/Desktop/Daily_Work_load/Github"
)

BRIDGING_PAPER = (
    "bridging-the-gap-between-data-reproduction-and-prediction-"
    "the-impact-of-feature-selection-and-cross-validation.pdf"
)

SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".md",
    ".txt",
    ".py",
    ".ipynb",
    ".html",
    ".htm",
}

IGNORED_DIRECTORIES = {
    ".git",
    ".github",
    ".ipynb_checkpoints",
    "__pycache__",
    ".venv",
    "venv",
    "env",
    "node_modules",
    "build",
    "dist",
}

MAX_FILE_SIZE_BYTES = 2_000_000

In [ ]:
DOCUMENT_TYPES = {
    "main.pdf": "thesis",
    BRIDGING_PAPER: "research_paper",
    "Milad Saeedi - ML.pdf": "cv_machine_learning",
    "Milad Saeedi - GEO.pdf": "cv_geospatial",
}

TWO_COLUMN_FILES = {
    BRIDGING_PAPER,
}

EXCLUDED_PAGE_RANGES = {
    "main.pdf": [
        (4, 12, "Contents, list of figures, and list of tables"),
        (104, 123, "Bibliography"),
    ],
    BRIDGING_PAPER: [
        (11, 12, "References"),
    ],
}

In [ ]:
print("PDF/data directory:", DATA_DIR.resolve())
print("Data directory exists:", DATA_DIR.exists())

print()
print("GitHub directory:", GITHUB_DIR.resolve())
print("GitHub directory exists:", GITHUB_DIR.exists())

## 3. Define the document structure

All source types—PDFs, Markdown files, notebooks, Python files, HTML pages, and text files—will be converted into the same `Document` format.

Using one shared structure keeps the downstream pipeline simple:

- filtering
- chunking
- embedding
- retrieval
- citation generation

In [ ]:
@dataclass
class Document:
    text: str
    source: str
    file_type: str

    page: int | None = None
    document_type: str | None = None
    page_header: str | None = None

    repository: str | None = None
    relative_path: str | None = None
    section: str | None = None

In [ ]:
def clean_text(text: str) -> str:
    """
    Normalize text extracted from different file types.
    """

    text = text.replace("\x00", " ")

    # Normalize spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces before line breaks
    text = re.sub(r" +\n", "\n", text)

    # Reduce repeated blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
MAX_TEXT_FILE_SIZE_BYTES = 2_000_000


def should_ignore_path(path: Path) -> bool:
    """
    Ignore noisy directories and unusually large non-PDF files.

    PDFs are allowed to be larger because they are processed page by page.
    """

    if any(
        part in IGNORED_DIRECTORIES
        for part in path.parts
    ):
        return True

    if (
        path.is_file()
        and path.suffix.lower() != ".pdf"
        and path.stat().st_size > MAX_TEXT_FILE_SIZE_BYTES
    ):
        return True

    return False

In [ ]:
def find_repository_root(path: Path) -> Path | None:
    """
    Find the nearest parent directory containing a .git folder.
    """

    for parent in [path.parent, *path.parents]:
        if (parent / ".git").exists():
            return parent

    return None

In [ ]:
print(
    clean_text(
        "This   text has     extra spaces.\n\n\n\n"
        "It also has repeated blank lines."
    )
)

## 4. Load text-based project files

Markdown, Python, plain-text, and HTML files are converted into the shared `Document` format.

For files inside cloned Git repositories, the loader also records:

- repository name
- relative file path

This metadata will later be used for readable source citations.

In [ ]:
def get_source_metadata(
    path: Path,
    root_directory: Path,
) -> dict:
    """
    Build common metadata for local and GitHub files.
    """

    repository_root = find_repository_root(path)

    if repository_root is not None:
        return {
            "document_type": "github_project",
            "repository": repository_root.name,
            "relative_path": str(
                path.relative_to(repository_root)
            ),
        }

    return {
        "document_type": "local_document",
        "repository": None,
        "relative_path": str(
            path.relative_to(root_directory)
        ),
    }

In [ ]:
def load_text_file(
    path: Path,
    root_directory: Path,
) -> list[Document]:
    """
    Load Markdown, Python, or plain-text files.
    """

    text = path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    text = clean_text(text)

    if not text:
        return []

    metadata = get_source_metadata(
        path=path,
        root_directory=root_directory,
    )

    return [
        Document(
            text=text,
            source=path.name,
            file_type=path.suffix.lower().lstrip("."),
            document_type=metadata["document_type"],
            repository=metadata["repository"],
            relative_path=metadata["relative_path"],
        )
    ]

In [ ]:
def load_html_file(
    path: Path,
    root_directory: Path,
) -> list[Document]:
    """
    Extract readable text from an HTML file.
    """

    html = path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    for tag in soup(
        ["script", "style", "nav", "footer"]
    ):
        tag.decompose()

    text = clean_text(
        soup.get_text("\n")
    )

    if not text:
        return []

    metadata = get_source_metadata(
        path=path,
        root_directory=root_directory,
    )

    return [
        Document(
            text=text,
            source=path.name,
            file_type="html",
            document_type=metadata["document_type"],
            repository=metadata["repository"],
            relative_path=metadata["relative_path"],
        )
    ]

Test

In [ ]:
readme_files = list(
    GITHUB_DIR.rglob("README.md")
)

print("README files found:", len(readme_files))

for path in readme_files[:10]:
    print(path)

In [ ]:
sample_readme = load_text_file(
    path=readme_files[0],
    root_directory=GITHUB_DIR,
)

sample_document = sample_readme[0]

print("Source:", sample_document.source)
print("Repository:", sample_document.repository)
print("Relative path:", sample_document.relative_path)
print("File type:", sample_document.file_type)
print()
print(sample_document.text[:1000])

## 5. Load Jupyter notebooks

Notebook cells are grouped into meaningful sections instead of being treated as isolated cells.

A Markdown heading starts a new section, and the code cells that follow it are attached to that section. This preserves both the explanation and its implementation while reducing retrieval noise.

Notebook outputs are ignored because they may contain large arrays, logs, plots, widgets, or repeated results.

In [ ]:
def extract_markdown_title(markdown_text: str) -> str | None:
    """
    Return the first Markdown heading in a cell.
    """

    for line in markdown_text.splitlines():
        match = re.match(
            r"^\s*#{1,6}\s+(.+?)\s*$",
            line,
        )

        if match:
            return match.group(1).strip()

    return None

In [ ]:
def is_useful_code(code_text: str) -> bool:
    """
    Exclude very small or display-only code cells.
    """

    code_text = code_text.strip()

    if not code_text:
        return False

    lines = [
        line.strip()
        for line in code_text.splitlines()
        if line.strip()
    ]

    trivial_prefixes = (
        "print(",
        "display(",
        "plt.show(",
        "%matplotlib",
    )

    meaningful_lines = [
        line
        for line in lines
        if not line.startswith(trivial_prefixes)
    ]

    return len(meaningful_lines) >= 2

In [ ]:
def load_notebook_file(
    path: Path,
    root_directory: Path,
) -> list[Document]:
    """
    Group Markdown explanations with their following code cells.
    """

    try:
        notebook = nbformat.read(
            path,
            as_version=4,
        )
    except Exception as error:
        print(f"Could not read {path}: {error}")
        return []

    metadata = get_source_metadata(
        path=path,
        root_directory=root_directory,
    )

    notebook_documents = []

    current_title = None
    current_parts = []
    section_number = 0

    def save_current_section() -> None:
        nonlocal current_parts
        nonlocal section_number

        combined_text = "\n\n".join(
            current_parts
        ).strip()

        if not combined_text:
            return

        section_number += 1

        notebook_documents.append(
            Document(
                text=combined_text,
                source=path.name,
                file_type="ipynb",
                document_type=metadata["document_type"],
                repository=metadata["repository"],
                relative_path=metadata["relative_path"],
                section=(
                    current_title
                    or f"Notebook section {section_number}"
                ),
            )
        )

        current_parts = []

    for cell in notebook.cells:
        if cell.cell_type == "markdown":
            markdown_text = clean_text(
                cell.source or ""
            )

            if not markdown_text:
                continue

            new_title = extract_markdown_title(
                markdown_text
            )

            if new_title and current_parts:
                save_current_section()

            if new_title:
                current_title = new_title

            current_parts.append(
                f"Explanation:\n{markdown_text}"
            )

        elif cell.cell_type == "code":
            code_text = clean_text(
                cell.source or ""
            )

            if not is_useful_code(code_text):
                continue

            current_parts.append(
                f"Python implementation:\n{code_text}"
            )

    save_current_section()

    return notebook_documents

Test

In [ ]:
notebook_files = [
    path
    for path in GITHUB_DIR.rglob("*.ipynb")
    if not should_ignore_path(path)
]

print("Notebook files found:", len(notebook_files))

for path in notebook_files[:10]:
    print(path)

In [ ]:
sample_notebook_documents = load_notebook_file(
    path=notebook_files[0],
    root_directory=GITHUB_DIR,
)

print(
    "Sections created:",
    len(sample_notebook_documents),
)

for document in sample_notebook_documents[:5]:
    print("=" * 100)
    print("Repository:", document.repository)
    print("File:", document.relative_path)
    print("Section:", document.section)
    print()
    print(document.text[:1000])
    print()

## 6. Load PDF documents

PDF documents require additional processing because their layouts and page structures may differ.

This loader supports:

- Single-column PDFs, such as the thesis and CVs
- Two-column journal articles
- Repeated running-header detection
- Repeated footer detection
- Removal of standalone page numbers
- Preservation of useful chapter and section metadata
- Page-level source citations

Explicitly excluded pages, such as contents pages and bibliographies, will be removed later before chunking.

In [ ]:
def is_page_number_only(text: str) -> bool:
    """Return True when a line contains only a page number."""

    text = text.strip()

    patterns = [
        r"^\d+$",
        r"^page\s+\d+$",
        r"^\d+\s+of\s+\d+$",
        r"^\d+\s*/\s*\d+$",
        r"^[ivxlcdm]+$",
    ]

    return any(
        re.fullmatch(pattern, text, flags=re.IGNORECASE)
        for pattern in patterns
    )


def is_chapter_header(text: str) -> bool:
    """Identify chapter, appendix, or section headings."""

    patterns = [
        r"^CHAPTER\s+\d+\b",
        r"^APPENDIX\s+[A-Z0-9]+\b",
        r"^SECTION\s+\d+\b",
    ]

    return any(
        re.match(pattern, text.strip(), flags=re.IGNORECASE)
        for pattern in patterns
    )

In [ ]:
def normalize_margin_text(text: str) -> str:
    """Normalize possible header or footer text."""

    text = text.strip()

    if not text or is_page_number_only(text):
        return ""

    text = text.lower()

    # Normalize changing numbers such as page numbers and years.
    text = re.sub(r"\d+", "<number>", text)

    # Normalize spacing.
    text = re.sub(r"\s+", " ", text)

    return text.strip(" |–—-.,:;()[]")

In [ ]:
def get_page_lines(page: fitz.Page) -> list[dict]:
    """Extract text lines and their coordinates."""

    lines = []
    page_data = page.get_text("dict")

    for block in page_data.get("blocks", []):
        if block.get("type") != 0:
            continue

        for line in block.get("lines", []):
            line_text = "".join(
                span.get("text", "")
                for span in line.get("spans", [])
            ).strip()

            if not line_text:
                continue

            x0, y0, x1, y1 = line["bbox"]

            lines.append(
                {
                    "text": line_text,
                    "x0": float(x0),
                    "y0": float(y0),
                    "x1": float(x1),
                    "y1": float(y1),
                }
            )

    return lines

In [ ]:
def detect_repeated_margins(
    pdf: fitz.Document,
    header_ratio: float = 0.12,
    footer_ratio: float = 0.08,
    repeat_fraction: float = 0.35,
) -> tuple[set[str], set[str]]:
    """
    Detect text repeatedly appearing in page headers and footers.
    """

    header_counts = Counter()
    footer_counts = Counter()

    total_pages = len(pdf)

    for page in pdf:
        page_height = page.rect.height

        page_headers = set()
        page_footers = set()

        for line in get_page_lines(page):
            normalized = normalize_margin_text(line["text"])

            if not normalized:
                continue

            if line["y1"] <= page_height * header_ratio:
                page_headers.add(normalized)

            if line["y0"] >= page_height * (1 - footer_ratio):
                page_footers.add(normalized)

        header_counts.update(page_headers)
        footer_counts.update(page_footers)

    minimum_pages = max(
        2,
        int(np.ceil(total_pages * repeat_fraction)),
    )

    repeated_headers = {
        text
        for text, count in header_counts.items()
        if count >= minimum_pages
    }

    repeated_footers = {
        text
        for text, count in footer_counts.items()
        if count >= minimum_pages
    }

    return repeated_headers, repeated_footers

In [ ]:
def get_page_header(
    page: fitz.Page,
    repeated_headers: set[str],
    header_ratio: float = 0.12,
) -> str | None:
    """Return useful running or chapter headers for one page."""

    page_height = page.rect.height
    header_lines = []

    for line in sorted(
        get_page_lines(page),
        key=lambda item: (item["y0"], item["x0"]),
    ):
        if line["y1"] > page_height * header_ratio:
            continue

        text = line["text"].strip()

        if is_page_number_only(text):
            continue

        normalized = normalize_margin_text(text)

        if (
            normalized in repeated_headers
            or is_chapter_header(text)
        ):
            header_lines.append(text)

    header_lines = list(dict.fromkeys(header_lines))

    if not header_lines:
        return None

    return " | ".join(header_lines)

In [ ]:
def extract_single_column_page(
    page: fitz.Page,
    repeated_headers: set[str],
    repeated_footers: set[str],
    header_ratio: float = 0.12,
    footer_ratio: float = 0.08,
) -> tuple[str, str | None]:
    """Extract and clean one single-column PDF page."""

    page_height = page.rect.height

    page_header = get_page_header(
        page=page,
        repeated_headers=repeated_headers,
        header_ratio=header_ratio,
    )

    retained_lines = []

    for line in sorted(
        get_page_lines(page),
        key=lambda item: (item["y0"], item["x0"]),
    ):
        text = line["text"].strip()

        if not text:
            continue

        in_header = line["y1"] <= page_height * header_ratio
        in_footer = line["y0"] >= page_height * (1 - footer_ratio)

        normalized = normalize_margin_text(text)

        repeated_header = (
            in_header
            and normalized
            and normalized in repeated_headers
        )

        repeated_footer = (
            in_footer
            and normalized
            and normalized in repeated_footers
        )

        margin_page_number = (
            is_page_number_only(text)
            and (in_header or in_footer)
        )

        chapter_running_header = (
            in_header
            and is_chapter_header(text)
        )

        if (
            repeated_header
            or repeated_footer
            or margin_page_number
            or chapter_running_header
        ):
            continue

        retained_lines.append(text)

    page_text = clean_text(
        "\n".join(retained_lines)
    )

    return page_text, page_header

In [ ]:
def get_page_blocks(page: fitz.Page) -> list[dict]:
    """Extract text blocks and their coordinates."""

    blocks = []

    for block in page.get_text("blocks", sort=False):
        x0, y0, x1, y1, text, block_number, block_type = block[:7]

        if block_type != 0:
            continue

        blocks.append(
            {
                "text": text,
                "x0": float(x0),
                "y0": float(y0),
                "x1": float(x1),
                "y1": float(y1),
                "block_number": block_number,
            }
        )

    return blocks

In [ ]:
def clean_pdf_block(
    block: dict,
    page_height: float,
    repeated_headers: set[str],
    repeated_footers: set[str],
    header_ratio: float = 0.12,
    footer_ratio: float = 0.08,
) -> dict | None:
    """Remove headers, footers, and page numbers from a text block."""

    in_header = block["y1"] <= page_height * header_ratio
    in_footer = block["y0"] >= page_height * (1 - footer_ratio)

    retained_lines = []

    for raw_line in block["text"].splitlines():
        line = raw_line.strip()

        if not line:
            continue

        normalized = normalize_margin_text(line)

        repeated_header = (
            in_header
            and normalized
            and normalized in repeated_headers
        )

        repeated_footer = (
            in_footer
            and normalized
            and normalized in repeated_footers
        )

        margin_page_number = (
            is_page_number_only(line)
            and (in_header or in_footer)
        )

        chapter_running_header = (
            in_header
            and is_chapter_header(line)
        )

        if (
            repeated_header
            or repeated_footer
            or margin_page_number
            or chapter_running_header
        ):
            continue

        retained_lines.append(line)

    block_text = clean_text(
        "\n".join(retained_lines)
    )

    if not block_text:
        return None

    cleaned_block = block.copy()
    cleaned_block["text"] = block_text

    return cleaned_block

In [ ]:
def order_two_column_blocks(
    blocks: list[dict],
    page_width: float,
) -> list[dict]:
    """Order full-width, left-column, and right-column blocks."""

    if not blocks:
        return []

    page_center = page_width / 2

    full_width_blocks = []
    column_blocks = []

    for block in blocks:
        block_width = block["x1"] - block["x0"]

        crosses_center = (
            block["x0"] < page_center - 15
            and block["x1"] > page_center + 15
        )

        is_wide = block_width >= page_width * 0.55

        if crosses_center or is_wide:
            full_width_blocks.append(block)
        else:
            column_blocks.append(block)

    full_width_blocks.sort(
        key=lambda item: (item["y0"], item["x0"])
    )

    remaining = list(column_blocks)
    ordered = []

    def add_columns(section_blocks: list[dict]) -> None:
        left = []
        right = []

        for block in section_blocks:
            block_center = (
                block["x0"] + block["x1"]
            ) / 2

            if block_center < page_center:
                left.append(block)
            else:
                right.append(block)

        left.sort(key=lambda item: (item["y0"], item["x0"]))
        right.sort(key=lambda item: (item["y0"], item["x0"]))

        ordered.extend(left)
        ordered.extend(right)

    for full_width_block in full_width_blocks:
        full_width_center_y = (
            full_width_block["y0"] + full_width_block["y1"]
        ) / 2

        blocks_above = [
            block
            for block in remaining
            if (
                block["y0"] + block["y1"]
            ) / 2 < full_width_center_y
        ]

        add_columns(blocks_above)

        remaining = [
            block
            for block in remaining
            if block not in blocks_above
        ]

        ordered.append(full_width_block)

    add_columns(remaining)

    return ordered

In [ ]:
def extract_two_column_page(
    page: fitz.Page,
    repeated_headers: set[str],
    repeated_footers: set[str],
) -> tuple[str, str | None]:
    """Extract one two-column journal page."""

    page_header = get_page_header(
        page=page,
        repeated_headers=repeated_headers,
    )

    cleaned_blocks = []

    for block in get_page_blocks(page):
        cleaned_block = clean_pdf_block(
            block=block,
            page_height=page.rect.height,
            repeated_headers=repeated_headers,
            repeated_footers=repeated_footers,
        )

        if cleaned_block is not None:
            cleaned_blocks.append(cleaned_block)

    ordered_blocks = order_two_column_blocks(
        blocks=cleaned_blocks,
        page_width=page.rect.width,
    )

    page_text = clean_text(
        "\n\n".join(
            block["text"]
            for block in ordered_blocks
        )
    )

    return page_text, page_header

In [ ]:
def load_pdf_file(
    path: Path,
    root_directory: Path,
) -> list[Document]:
    """
    Load one PDF page by page using the configured layout.
    """

    documents = []
    is_two_column = path.name in TWO_COLUMN_FILES

    with fitz.open(path) as pdf:
        repeated_headers, repeated_footers = (
            detect_repeated_margins(pdf)
        )

        for page_index, page in enumerate(pdf, start=1):
            try:
                if is_two_column:
                    page_text, page_header = extract_two_column_page(
                        page=page,
                        repeated_headers=repeated_headers,
                        repeated_footers=repeated_footers,
                    )
                else:
                    page_text, page_header = extract_single_column_page(
                        page=page,
                        repeated_headers=repeated_headers,
                        repeated_footers=repeated_footers,
                    )

                if not page_text:
                    print(
                        f"Warning: no text extracted from "
                        f"{path.name}, page {page_index}"
                    )
                    continue

                documents.append(
                    Document(
                        text=page_text,
                        source=path.name,
                        file_type="pdf",
                        page=page_index,
                        document_type=DOCUMENT_TYPES.get(
                            path.name,
                            "pdf_document",
                        ),
                        page_header=page_header,
                        relative_path=str(
                            path.relative_to(root_directory)
                        ),
                    )
                )

            except Exception as error:
                print(
                    f"Error reading {path.name}, "
                    f"page {page_index}: {error}"
                )

    return documents

## 7. Load all supported files

The project uses two source folders:

- `DATA_DIR` for the thesis, papers, CVs, and other local documents
- `GITHUB_DIR` for cloned GitHub repositories

Both folders are scanned recursively. Each supported file is sent to the appropriate loader and converted into the shared `Document` format.

In [ ]:
def load_file(
    path: Path,
    root_directory: Path,
) -> list[Document]:
    """
    Route a file to the correct loader based on its extension.
    """

    suffix = path.suffix.lower()

    if suffix == ".pdf":
        return load_pdf_file(
            path=path,
            root_directory=root_directory,
        )

    if suffix == ".ipynb":
        return load_notebook_file(
            path=path,
            root_directory=root_directory,
        )

    if suffix in {".html", ".htm"}:
        return load_html_file(
            path=path,
            root_directory=root_directory,
        )

    if suffix in {".md", ".txt", ".py"}:
        return load_text_file(
            path=path,
            root_directory=root_directory,
        )

    return []

In [ ]:
def load_directory(
    root_directory: Path,
    allowed_extensions: set[str],
) -> list[Document]:
    """
    Recursively load only the allowed file types from one directory.
    """

    documents = []
    loaded_files = 0
    failed_files = []

    for path in sorted(root_directory.rglob("*")):
        if not path.is_file():
            continue

        if should_ignore_path(path):
            continue

        if path.suffix.lower() not in allowed_extensions:
            continue

        try:
            file_documents = load_file(
                path=path,
                root_directory=root_directory,
            )

            documents.extend(file_documents)
            loaded_files += 1

            print(
                f"Loaded: {path.relative_to(root_directory)} "
                f"→ {len(file_documents)} section(s)"
            )

        except Exception as error:
            failed_files.append(
                {
                    "file": str(path),
                    "error": f"{type(error).__name__}: {error}",
                }
            )

            print(
                f"Failed: {path.relative_to(root_directory)} "
                f"→ {type(error).__name__}: {error}"
            )

    print()
    print("Supported files loaded:", loaded_files)
    print("Document sections created:", len(documents))
    print("Failed files:", len(failed_files))

    return documents

In [ ]:
LOCAL_EXTENSIONS = {
    ".pdf",
    ".md",
    ".txt",
    ".html",
    ".htm",
}

GITHUB_EXTENSIONS = {
    ".md",
    ".ipynb",
    ".py",
    ".html",
    ".htm",
}

print("=" * 100)
print("Loading local documents")

local_documents = load_directory(
    root_directory=DATA_DIR,
    allowed_extensions=LOCAL_EXTENSIONS,
)

print()
print("=" * 100)
print("Loading cloned GitHub repositories")



github_documents = load_directory(
    root_directory=GITHUB_DIR,
    allowed_extensions=GITHUB_EXTENSIONS,
)

documents = local_documents + github_documents

## 8. Exclude confirmed low-value PDF pages

Some PDF pages contain content that is not useful for retrieval, such as:

- Tables of contents
- Lists of figures and tables
- Bibliographies
- Reference sections

Only explicitly confirmed page ranges are excluded. Other source types, including notebooks, README files, Python files, and HTML documents, remain unchanged.

In [ ]:
def get_exclusion_reason(document: Document) -> str | None:
    """
    Return the reason for excluding a configured PDF page.
    """

    if document.file_type != "pdf" or document.page is None:
        return None

    page_ranges = EXCLUDED_PAGE_RANGES.get(
        document.source,
        [],
    )

    for start_page, end_page, reason in page_ranges:
        if start_page <= document.page <= end_page:
            return reason

    return None

In [ ]:
filtered_documents = []
excluded_documents = []

for document in documents:
    exclusion_reason = get_exclusion_reason(document)

    if exclusion_reason is None:
        filtered_documents.append(document)
    else:
        excluded_documents.append(
            {
                "source": document.source,
                "page": document.page,
                "reason": exclusion_reason,
            }
        )

In [ ]:
print("Original document sections:", len(documents))
print("Excluded PDF pages:", len(excluded_documents))
print("Remaining document sections:", len(filtered_documents))

In [ ]:
excluded_df = pd.DataFrame(excluded_documents)

if excluded_df.empty:
    print("No documents were excluded.")
else:
    display(
        excluded_df.sort_values(
            ["source", "page"]
        )
    )

## 9. Chunk the documents

The remaining PDF pages, README files, notebook sections, Python files, HTML files, and text documents are split into smaller overlapping chunks.

Chunking improves retrieval because each embedding represents a focused passage rather than an entire long document.

The overlap preserves context when an important idea crosses a chunk boundary.

In [ ]:
@dataclass
class Chunk:
    text: str
    source: str
    file_type: str
    chunk_id: str

    page: int | None = None
    document_type: str | None = None
    page_header: str | None = None

    repository: str | None = None
    relative_path: str | None = None
    section: str | None = None

In [ ]:
def split_text(
    text: str,
    chunk_size: int = 1000,
    overlap: int = 200,
) -> list[str]:
    """
    Split text into overlapping chunks while preferring
    paragraph, sentence, or line boundaries.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError(
            "overlap must be non-negative and smaller than chunk_size"
        )

    text_chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        candidate = text[start:end]

        if end < len(text):
            split_position = max(
                candidate.rfind("\n\n"),
                candidate.rfind(". "),
                candidate.rfind("\n"),
            )

            if split_position > chunk_size // 2:
                end = start + split_position + 1
                candidate = text[start:end]

        candidate = candidate.strip()

        if candidate:
            text_chunks.append(candidate)

        if end >= len(text):
            break

        start = end - overlap

    return text_chunks

In [ ]:
def create_chunks(
    documents: list[Document],
    chunk_size: int = 1000,
    overlap: int = 200,
) -> list[Chunk]:
    """
    Split all documents into searchable chunks while
    preserving source metadata.
    """

    chunks = []

    for document_index, document in enumerate(documents):
        document_chunks = split_text(
            text=document.text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_index, chunk_text in enumerate(document_chunks):
            chunks.append(
                Chunk(
                    text=chunk_text,
                    source=document.source,
                    file_type=document.file_type,
                    chunk_id=f"{document_index}-{chunk_index}",
                    page=document.page,
                    document_type=document.document_type,
                    page_header=document.page_header,
                    repository=document.repository,
                    relative_path=document.relative_path,
                    section=document.section,
                )
            )

    return chunks

### Build metadata-aware retrieval text

Repository names, filenames, notebook section titles, and PDF metadata can contain important retrieval clues.

These fields are added to the text used by FAISS and BM25, while the original chunk text remains unchanged for answer generation.

In [ ]:
def build_retrieval_text(chunk: Chunk) -> str:
    """
    Combine retrieval-relevant metadata with the original chunk text.
    """

    metadata_parts = []

    if chunk.repository:
        metadata_parts.append(
            f"Repository: {chunk.repository}"
        )

    if chunk.relative_path:
        metadata_parts.append(
            f"File: {chunk.relative_path}"
        )

    if chunk.section:
        metadata_parts.append(
            f"Section: {chunk.section}"
        )

    if chunk.document_type:
        metadata_parts.append(
            f"Document type: {chunk.document_type}"
        )

    if chunk.page_header:
        metadata_parts.append(
            f"Page header: {chunk.page_header}"
        )

    if chunk.page is not None:
        metadata_parts.append(
            f"Page: {chunk.page}"
        )

    metadata_text = "\n".join(metadata_parts)

    if metadata_text:
        return f"{metadata_text}\n\n{chunk.text}"

    return chunk.text

In [ ]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = create_chunks(
    documents=filtered_documents,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
)

print("Filtered document sections:", len(filtered_documents))
print("Total chunks created:", len(chunks))

In [ ]:
retrieval_texts = [
    build_retrieval_text(chunk)
    for chunk in chunks
]

print("Chunks:", len(chunks))
print("Retrieval texts:", len(retrieval_texts))

assert len(chunks) == len(retrieval_texts)

In [ ]:
github_example_index = next(
    index
    for index, chunk in enumerate(chunks)
    if chunk.repository is not None
)

print(retrieval_texts[github_example_index][:1200])

In [ ]:
chunk_lengths = pd.Series(
    [len(chunk.text) for chunk in chunks]
)

chunk_lengths.describe()

In [ ]:
for chunk in chunks[:5]:
    print("=" * 100)
    print("Chunk ID:", chunk.chunk_id)
    print("Source:", chunk.source)
    print("File type:", chunk.file_type)
    print("Page:", chunk.page)
    print("Repository:", chunk.repository)
    print("Relative path:", chunk.relative_path)
    print("Section:", chunk.section)
    print("Characters:", len(chunk.text))
    print()
    print(chunk.text[:1200])
    print()

In [ ]:
unexpected_chunks = []

for chunk in chunks:
    if chunk.file_type != "pdf" or chunk.page is None:
        continue

    for start_page, end_page, reason in EXCLUDED_PAGE_RANGES.get(
        chunk.source,
        [],
    ):
        if start_page <= chunk.page <= end_page:
            unexpected_chunks.append(
                {
                    "source": chunk.source,
                    "page": chunk.page,
                    "chunk_id": chunk.chunk_id,
                    "reason": reason,
                }
            )

print("Chunks from excluded PDF pages:", len(unexpected_chunks))

assert len(unexpected_chunks) == 0

## 10. Create embeddings and build the vector index

Each chunk is converted into a numerical embedding using a retrieval-focused Sentence Transformer model.

The embeddings are normalized and stored in a FAISS index. This allows the system to compare a user question with all document chunks and retrieve the most semantically relevant passages.

In [ ]:
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

In [ ]:
embeddings = embedding_model.encode(
    retrieval_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=16,
)

embeddings = np.asarray(
    embeddings,
    dtype="float32",
)

print("Embedding matrix shape:", embeddings.shape)

assert len(embeddings) == len(chunks)

In [ ]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Vectors stored:", index.ntotal)

assert index.ntotal == len(chunks)

In [ ]:
print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)

assert index.ntotal == len(chunks)

print("FAISS index created successfully.")

In [ ]:
QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
)


def encode_query(query: str) -> np.ndarray:
    """
    Encode a user question for semantic retrieval.
    """

    instructed_query = (
        QUERY_INSTRUCTION
        + query.strip()
    )

    query_embedding = embedding_model.encode(
        [instructed_query],
        normalize_embeddings=True,
    )

    return np.asarray(
        query_embedding,
        dtype="float32",
    )

In [ ]:
test_query = (
    "What are Milad Saeedi's main research contributions?"
)

query_embedding = encode_query(
    test_query
)

scores, indices = index.search(
    query_embedding,
    5,
)

print("Similarity scores:")
print(scores)

print()
print("Retrieved chunk indices:")
print(indices)

In [ ]:
for rank, (score, chunk_index) in enumerate(
    zip(scores[0], indices[0]),
    start=1,
):
    chunk = chunks[chunk_index]

    print("=" * 100)
    print(f"Result {rank}")
    print("Similarity score:", round(float(score), 3))
    print("Source:", chunk.source)
    print("Page:", chunk.page)
    print("Repository:", chunk.repository)
    print("Relative path:", chunk.relative_path)
    print("Section:", chunk.section)
    print()
    print(chunk.text[:1500])
    print()

## 11. Build a Hybrid Retrieval System (FAISS + BM25)

Semantic search is excellent at finding conceptually similar passages, but it may overlook chunks containing important technical keywords, abbreviations, repository names, or paper titles.

To improve retrieval quality, this project combines two complementary retrieval methods:

- **FAISS semantic search**, which retrieves chunks based on vector similarity.
- **BM25 keyword search**, which retrieves chunks containing important query terms.

The candidate results from both retrieval methods are merged into a single candidate set. Duplicate chunks are removed before passing the candidates to a cross-encoder reranker.

This hybrid approach improves retrieval for technical questions involving:

- Model names (e.g., XGBoost, YOLOv8, LoRA, GRPO)
- Repository names
- Paper titles
- Acronyms (e.g., UFP, BC, STCV)
- Function and class names
- Exact terminology that semantic retrieval may underweight

The overall retrieval pipeline is:

Question

↓

FAISS semantic retrieval

+

BM25 keyword retrieval

↓

Merge and deduplicate candidates

↓

Cross-encoder reranking

↓

Top-ranked context

↓

LLM answer generation

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize_for_bm25(text: str) -> list[str]:
    """
    Tokenize text for BM25 keyword retrieval.

    Keeps technical terms such as:
    YOLOv8, XGBoost, LoRA, GRPO, STCV, UFP, and file names.
    """

    return re.findall(
        r"\b[a-zA-Z0-9][a-zA-Z0-9_.+-]*\b",
        text.lower(),
    )

In [ ]:
tokenized_chunks = [
    tokenize_for_bm25(text)
    for text in retrieval_texts
]

bm25 = BM25Okapi(tokenized_chunks)

print(
    "BM25 corpus documents:",
    len(tokenized_chunks),
)

assert len(tokenized_chunks) == len(chunks)

In [ ]:
def retrieve_faiss_candidates(
    query: str,
    top_k: int = 50,
) -> list[dict]:
    """
    Retrieve candidate chunks using FAISS semantic search.
    """

    if top_k <= 0:
        raise ValueError("top_k must be positive")

    search_k = min(top_k, index.ntotal)

    query_embedding = encode_query(query)

    scores, indices = index.search(
        query_embedding,
        search_k,
    )

    results = []

    for rank, (score, chunk_index) in enumerate(
        zip(scores[0], indices[0]),
        start=1,
    ):
        if chunk_index < 0:
            continue

        results.append(
            {
                "chunk_index": int(chunk_index),
                "faiss_score": float(score),
                "faiss_rank": rank,
            }
        )

    return results

In [ ]:
def retrieve_bm25_candidates(
    query: str,
    top_k: int = 50,
) -> list[dict]:
    """
    Retrieve candidate chunks using BM25 keyword search.
    """

    if top_k <= 0:
        raise ValueError("top_k must be positive")

    query_tokens = tokenize_for_bm25(query)

    if not query_tokens:
        return []

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, chunk_index in enumerate(
        top_indices,
        start=1,
    ):
        score = float(scores[chunk_index])

        # Ignore candidates with no keyword relevance.
        if score <= 0:
            continue

        results.append(
            {
                "chunk_index": int(chunk_index),
                "bm25_score": score,
                "bm25_rank": rank,
            }
        )

    return results

In [ ]:
def merge_candidates(
    faiss_results: list[dict],
    bm25_results: list[dict],
) -> list[dict]:
    """
    Merge FAISS and BM25 results using chunk indices.

    A chunk retrieved by both methods appears only once.
    """

    merged = {}

    for result in faiss_results:
        chunk_index = result["chunk_index"]

        merged[chunk_index] = {
            "chunk_index": chunk_index,
            "faiss_score": result["faiss_score"],
            "faiss_rank": result["faiss_rank"],
            "bm25_score": None,
            "bm25_rank": None,
            "retrieved_by": {"faiss"},
        }

    for result in bm25_results:
        chunk_index = result["chunk_index"]

        if chunk_index not in merged:
            merged[chunk_index] = {
                "chunk_index": chunk_index,
                "faiss_score": None,
                "faiss_rank": None,
                "bm25_score": result["bm25_score"],
                "bm25_rank": result["bm25_rank"],
                "retrieved_by": {"bm25"},
            }
        else:
            merged[chunk_index]["bm25_score"] = result["bm25_score"]
            merged[chunk_index]["bm25_rank"] = result["bm25_rank"]
            merged[chunk_index]["retrieved_by"].add("bm25")

    merged_results = list(merged.values())

    for result in merged_results:
        result["retrieved_by"] = sorted(result["retrieved_by"])

    return merged_results

In [ ]:
def retrieve_hybrid_candidates(
    query: str,
    faiss_k: int = 80,
    bm25_k: int = 20,
) -> list[dict]:
    """
    Retrieve and merge candidates from FAISS and BM25.
    """

    faiss_results = retrieve_faiss_candidates(
        query=query,
        top_k=faiss_k,
    )

    bm25_results = retrieve_bm25_candidates(
        query=query,
        top_k=bm25_k,
    )

    merged_results = merge_candidates(
        faiss_results=faiss_results,
        bm25_results=bm25_results,
    )

    return merged_results

In [ ]:
QUERY_EXPANSIONS = {
    "gan": "generative adversarial network",
    "gans": "generative adversarial networks",
    "cnn": "convolutional neural network",
    "llm": "large language model",
    "llms": "large language models",
    "lora": "low-rank adaptation",
    "grpo": "group relative policy optimization",
    "rnn": "recurrent neural network",
    "lstm": "long short-term memory",
}


def expand_query(query: str) -> str:
    """Append full forms of common technical abbreviations."""

    words = re.findall(r"\b[\w-]+\b", query.lower())

    expansions = [
        QUERY_EXPANSIONS[word]
        for word in words
        if word in QUERY_EXPANSIONS
    ]

    if not expansions:
        return query

    return query + " " + " ".join(expansions)

In [ ]:
def enrich_candidates(
    candidates: list[dict],
) -> list[dict]:

    enriched_results = []

    for candidate in candidates:
        chunk_index = candidate["chunk_index"]
        chunk = chunks[chunk_index]

        enriched_results.append(
            {
                **candidate,
                "text": chunk.text,
                "retrieval_text": retrieval_texts[chunk_index],
                "source": chunk.source,
                "file_type": chunk.file_type,
                "page": chunk.page,
                "document_type": chunk.document_type,
                "page_header": chunk.page_header,
                "repository": chunk.repository,
                "relative_path": chunk.relative_path,
                "section": chunk.section,
                "chunk_id": chunk.chunk_id,
            }
        )

    return enriched_results

In [ ]:
test_query = (
    "Which projects or publications use YOLOv8 "
    "and computer vision?"
)

hybrid_candidates = retrieve_hybrid_candidates(
    query=test_query,
    faiss_k=20,
    bm25_k=20,
)

hybrid_candidates = enrich_candidates(
    hybrid_candidates
)

print("Unique hybrid candidates:", len(hybrid_candidates))

In [ ]:
for candidate in hybrid_candidates[:10]:
    print("=" * 100)
    print("Chunk ID:", candidate["chunk_id"])
    print("Retrieved by:", candidate["retrieved_by"])
    print("FAISS rank:", candidate["faiss_rank"])
    print("FAISS score:", candidate["faiss_score"])
    print("BM25 rank:", candidate["bm25_rank"])
    print("BM25 score:", candidate["bm25_score"])
    print("Source:", candidate["source"])
    print("Page:", candidate["page"])
    print("Repository:", candidate["repository"])
    print("Relative path:", candidate["relative_path"])
    print("Section:", candidate["section"])
    print()
    print(candidate["text"][:1200])
    print()

## 12. Rerank hybrid candidates with a cross-encoder

FAISS and BM25 retrieve a broad set of potentially relevant chunks.

The cross-encoder then evaluates each query–chunk pair directly and assigns a relevance score. This produces a more precise final ranking than using semantic or keyword retrieval alone.

The complete retrieval flow is:

Question

↓

FAISS candidates + BM25 candidates

↓

Merge and deduplicate

↓

Cross-encoder reranking

↓

Top final chunks

In [ ]:
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"

reranker = CrossEncoder(
    RERANKER_MODEL_NAME
)

print("Reranker loaded:", RERANKER_MODEL_NAME)

In [ ]:
def rerank_candidates(
    query: str,
    candidates: list[dict],
    top_k: int = 5,
) -> list[dict]:
    """
    Rerank hybrid retrieval candidates using a cross-encoder.
    """

    if top_k <= 0:
        raise ValueError("top_k must be positive")

    if not candidates:
        return []

    query_chunk_pairs = [
        [query, candidate["retrieval_text"]]
        for candidate in candidates
    ]

    reranker_scores = reranker.predict(
        query_chunk_pairs,
        show_progress_bar=False,
    )

    reranked_results = []

    for candidate, score in zip(
        candidates,
        reranker_scores,
    ):
        result = candidate.copy()
        result["reranker_score"] = float(score)
        reranked_results.append(result)

    reranked_results.sort(
        key=lambda result: result["reranker_score"],
        reverse=True,
    )

    return reranked_results[:top_k]

In [ ]:
def retrieve_with_reranking(
    query: str,
    top_k: int = 5,
    faiss_k: int = 50,
    bm25_k: int = 50,
) -> list[dict]:

    search_query = expand_query(query)

    hybrid_candidates = retrieve_hybrid_candidates(
        query=search_query,
        faiss_k=faiss_k,
        bm25_k=bm25_k,
    )

    enriched_candidates = enrich_candidates(
        hybrid_candidates
    )

    # Use the original user query for precise reranking.
    return rerank_candidates(
        query=query,
        candidates=enriched_candidates,
        top_k=top_k,
    )

In [ ]:
test_query = (
    "Which projects or publications use YOLOv8 "
    "and computer vision?"
)

results = retrieve_with_reranking(
    query=test_query,
    top_k=5,
    faiss_k=50,
    bm25_k=50,
)

In [ ]:
for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print(f"Result {rank}")
    print("Reranker score:", round(result["reranker_score"], 3))
    print("Retrieved by:", result["retrieved_by"])
    print("FAISS rank:", result["faiss_rank"])
    print("FAISS score:", result["faiss_score"])
    print("BM25 rank:", result["bm25_rank"])
    print("BM25 score:", result["bm25_score"])
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Repository:", result["repository"])
    print("Relative path:", result["relative_path"])
    print("Section:", result["section"])
    print()
    print(result["text"][:1500])
    print()

## 13. Build the context and grounded prompt

The final retrieved chunks are converted into a structured context for the language model.

Each chunk receives a readable source label containing information such as:

- PDF filename and page number
- GitHub repository
- Relative file path
- Notebook section
- Document type

The prompt instructs the language model to answer only from the retrieved context, cite its sources, and avoid inventing unsupported details.

In [ ]:
def format_source_location(result: dict) -> str:
    """
    Create a readable source label for PDF and GitHub content.
    """

    if result.get("repository"):
        location = f"GitHub repository: {result['repository']}"

        if result.get("relative_path"):
            location += f", file: {result['relative_path']}"

        if result.get("section"):
            location += f", section: {result['section']}"

        return location

    location = result.get("source", "Unknown source")

    if result.get("page") is not None:
        location += f", page {result['page']}"

    if result.get("document_type"):
        location += f", type: {result['document_type']}"

    return location

In [ ]:
def build_context(results: list[dict]) -> str:
    """
    Convert retrieved results into a citation-aware context block.
    """

    context_parts = []

    for source_number, result in enumerate(results, start=1):
        location = format_source_location(result)

        context_parts.append(
            f"[Source {source_number}: {location}]\n"
            f"{result['text']}"
        )

    return "\n\n".join(context_parts)

In [ ]:
context = build_context(results)

print(context[:6000])

In [ ]:
def remove_source_section(text: str) -> str:
    """Remove the displayed source list from a chatbot response."""

    marker = "\n\n---\n\n### Sources"
    return text.split(marker, 1)[0].strip()

In [ ]:
def format_chat_history(
    history,
    max_messages: int = 6,
) -> str:
    """
    Convert recent Gradio history into readable conversation text.

    Supports:
    - Newer Gradio dictionaries:
      {"role": "user", "content": "..."}
    - Older Gradio tuples:
      ("user message", "assistant response")

    Source lists are removed from assistant responses before
    they are added to the conversation context.
    """

    if not history:
        return ""

    lines = []

    for item in history[-max_messages:]:

        # Newer Gradio message format
        if isinstance(item, dict):
            role = item.get("role", "").strip().lower()
            content = item.get("content", "")

            if not isinstance(content, str):
                continue

            content = content.strip()

            if role == "assistant":
                content = remove_source_section(content)

            if content and role in {"user", "assistant"}:
                lines.append(
                    f"{role.capitalize()}: {content}"
                )

        # Older Gradio tuple/list format
        elif (
            isinstance(item, (list, tuple))
            and len(item) == 2
        ):
            user_message, assistant_message = item

            if isinstance(user_message, str):
                user_message = user_message.strip()

                if user_message:
                    lines.append(
                        f"User: {user_message}"
                    )

            if isinstance(assistant_message, str):
                assistant_message = remove_source_section(
                    assistant_message
                )

                if assistant_message:
                    lines.append(
                        f"Assistant: {assistant_message}"
                    )

    return "\n".join(lines)

In [ ]:
def rewrite_question_with_history(
    question: str,
    history,
    model_name: str = "qwen2.5:3b",
) -> str:
    """Rewrite a follow-up question as a standalone retrieval query."""

    history_text = format_chat_history(history)

    if not history_text:
        return question

    prompt = f"""
Rewrite the latest question as a standalone search query.

Use the conversation only to resolve references such as:
he, his, it, that project, that paper, or the thesis.

Do not answer the question.
Return only the rewritten query.

Conversation:
{history_text}

Latest question:
{question}

Standalone query:
""".strip()

    try:
        rewritten = generate_answer(
            prompt=prompt,
            model_name=model_name,
        ).strip()

        return rewritten or question

    except Exception:
        return question

In [ ]:
def build_prompt(
    question: str,
    results: list[dict],
    history=None,
) -> str:

    context = build_context(results)
    history_text = format_chat_history(history)

    if not history_text:
        history_text = "No previous conversation."

    return f"""
You are ResearchGPT, a research and portfolio assistant for Milad Saeedi.

Use the retrieved context as the factual source of truth.

Instructions:
1. Answer the latest question directly.
2. Use conversation history only to resolve follow-up references.
3. Cite factual claims using [Source 1], [Source 2], and so on.
4. Do not invent publications, methods, results, skills, or projects.
5. If the retrieved context is insufficient, say so clearly.
6. Do not treat conversation history as evidence unless supported by a source.
7. Contact information may be provided only when explicitly requested.

Recent conversation:
{history_text}

Retrieved context:
{context}

Latest question:
{question}

Answer:
""".strip()

In [ ]:
question = (
    "Which projects or publications use YOLOv8 "
    "and computer vision?"
)

results = retrieve_with_reranking(
    query=question,
    top_k=5,
    faiss_k=50,
    bm25_k=50,
)

prompt = build_prompt(
    question=question,
    results=results,
)

print(prompt[:8000])

## 14. Generate grounded answers

The retrieved and reranked context is now sent to a language model.

The model receives:

- the user’s question
- the most relevant retrieved chunks
- source labels for citation
- instructions to avoid unsupported claims

This completes the Retrieval-Augmented Generation pipeline:

Question

↓

Hybrid retrieval with FAISS and BM25

↓

Cross-encoder reranking

↓

Citation-aware prompt

↓

Language-model answer

In [ ]:
def generate_answer(
    prompt: str,
    model_name: str = "qwen2.5:3b",
) -> str:
    """
    Generate a grounded answer using a local Ollama model.
    """

    response = chat(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        options={
            "temperature": 0.2,
        },
    )

    return response.message.content.strip()

In [ ]:
gan_query = (
    "Summarize Milad Saeedi's GitHub GANs projects."
)

gan_results = retrieve_with_reranking(
    query=gan_query,
    top_k=10,
    faiss_k=100,
    bm25_k=100,
)

for rank, result in enumerate(
    gan_results,
    start=1,
):
    print("=" * 100)
    print("Rank:", rank)
    print("Repository:", result.get("repository"))
    print("Path:", result.get("relative_path"))
    print("Section:", result.get("section"))
    print("Reranker score:", result.get("reranker_score"))
    print()
    print(result["text"][:700])

In [ ]:
def answer_question(
    question: str,
    history=None,
    top_k: int = 5,
    faiss_k: int = 50,
    bm25_k: int = 50,
    model_name: str = "qwen2.5:3b",
) -> tuple[str, list[dict]]:

    question = question.strip()

    if not question:
        return "Please enter a question.", []

    retrieval_query = rewrite_question_with_history(
        question=question,
        history=history,
        model_name=model_name,
    )

    results = retrieve_with_reranking(
        query=retrieval_query,
        top_k=top_k,
        faiss_k=faiss_k,
        bm25_k=bm25_k,
    )

    if not results:
        return (
            "I could not find relevant information in the knowledge base.",
            [],
        )

    prompt = build_prompt(
        question=question,
        results=results,
        history=history,
    )

    answer = generate_answer(
        prompt=prompt,
        model_name=model_name,
    )

    return answer, results

In [ ]:
question = (
    "Which projects or publications use YOLOv8 "
    "and computer vision?"
)

answer, sources = answer_question(
    question=question,
    top_k=10,
    faiss_k=50,
    bm25_k=50,
)

print(answer)



In [ ]:
from IPython.display import Markdown, display

display(
    Markdown(answer)
)

In [ ]:
for rank, source in enumerate(sources, start=1):
    print("=" * 100)
    print(f"Source {rank}")
    print("Location:", format_source_location(source))
    print("Reranker score:", round(source["reranker_score"], 3))
    print("Retrieved by:", source["retrieved_by"])
    print()
    print(source["text"][:1200])
    print()

In [ ]:
test_questions = [
    "What are Milad Saeedi's main research contributions?",
    "Summarize Milad Saeedi's GitHub projects.",
    "Summarize Milad Saeedi's GitHub GAN projects.",
    "Which projects use PyTorch?",
    "What experience does Milad have with LoRA and GRPO?",
    "How was mobile air pollution monitoring used?",
    "Which work involved computer vision and 360-degree imagery?",
]

In [ ]:

qa_results = []

for question in test_questions:
    print("=" * 100)
    print("Question:", question)
    print()
    
    answer, sources = answer_question(
        question=question,
        top_k=5,
        faiss_k=30,
        bm25_k=30,
    )
    print(answer)
    print()
    
    qa_results.append(
        {
            "question": question,
            "answer": answer,
            "sources": sources,
        }
    )

In [ ]:
for result in qa_results:
    display(
        Markdown(
            f"""
---

## Question

{result["question"]}

## Answer

{result["answer"]}
"""
        )
    )

## 15. Save retrieval artifacts

The completed FAISS index, chunk metadata, retrieval texts, and model configuration are saved so the deployed application can start without rebuilding the knowledge base.

In [ ]:
from dataclasses import asdict
import json

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
faiss.write_index(
    index,
    str(ARTIFACT_DIR / "research_index.faiss"),
)

In [ ]:
with open(
    ARTIFACT_DIR / "chunks.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        [asdict(chunk) for chunk in chunks],
        file,
        ensure_ascii=False,
        indent=2,
    )

In [ ]:
with open(
    ARTIFACT_DIR / "retrieval_texts.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        retrieval_texts,
        file,
        ensure_ascii=False,
    )

In [ ]:
index_config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "reranker_model": RERANKER_MODEL_NAME,
    "embedding_dimension": embeddings.shape[1],
    "number_of_chunks": len(chunks),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "query_instruction": QUERY_INSTRUCTION,
}

with open(
    ARTIFACT_DIR / "index_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        index_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

## 16. Verify saved artifacts

In [ ]:
loaded_index = faiss.read_index(
    str(ARTIFACT_DIR / "research_index.faiss")
)

with open(
    ARTIFACT_DIR / "chunks.json",
    "r",
    encoding="utf-8",
) as file:
    loaded_chunks = [
        Chunk(**item)
        for item in json.load(file)
    ]

with open(
    ARTIFACT_DIR / "retrieval_texts.json",
    "r",
    encoding="utf-8",
) as file:
    loaded_retrieval_texts = json.load(file)

loaded_bm25 = BM25Okapi(
    [
        tokenize_for_bm25(text)
        for text in loaded_retrieval_texts
    ]
)

assert loaded_index.ntotal == len(loaded_chunks)
assert len(loaded_chunks) == len(loaded_retrieval_texts)

print("Artifacts loaded successfully.")
print("Vectors:", loaded_index.ntotal)
print("Chunks:", len(loaded_chunks))

In [ ]:
with open(
    ARTIFACT_DIR / "index_config.json",
    "r",
    encoding="utf-8",
) as file:
    loaded_config = json.load(file)

assert loaded_config["number_of_chunks"] == len(loaded_chunks)
assert loaded_config["embedding_dimension"] == loaded_index.d
assert loaded_config["embedding_model"] == EMBEDDING_MODEL_NAME
assert loaded_config["reranker_model"] == RERANKER_MODEL_NAME

print("Artifact configuration validated.")

## 17. Build the Gradio chatbot

The Gradio interface provides an interactive chatbot for exploring Milad Saeedi’s:

- Research and publications
- PhD thesis
- Machine learning and computer vision experience
- Generative AI and LLM projects
- GitHub repositories and notebooks

Each response is generated using the RAG pipeline and includes the retrieved sources.

In [ ]:
def format_source(source: dict) -> str:
    """
    Create a readable source label for PDF and GitHub content.
    """

    if source.get("repository"):
        location = f"GitHub: {source['repository']}"

        if source.get("relative_path"):
            location += f" / {source['relative_path']}"

        if source.get("section"):
            location += f" — {source['section']}"

        return location

    location = source.get("source", "Unknown source")

    if source.get("page") is not None:
        location += f", page {source['page']}"

    return location

In [ ]:
def format_sources(sources: list[dict]) -> str:
    """
    Format retrieved sources as Markdown.
    """

    if not sources:
        return "No supporting sources were retrieved."

    lines = ["### Sources"]

    seen_locations = set()

    for source in sources:
        location = format_source(source)

        # Avoid displaying the same source more than once.
        if location in seen_locations:
            continue

        seen_locations.add(location)
        lines.append(f"- {location}")

    return "\n".join(lines)

In [ ]:
test_answer, test_sources = answer_question(
    question="Summarize Milad Saeedi's GitHub GAN projects.",
    top_k=8,
    faiss_k=50,
    bm25_k=20,
)

display(Markdown(test_answer))
display(Markdown(format_sources(test_sources)))

In [ ]:
def research_chat(
    message: str,
    history: list[dict],
) -> str:
    """
    Answer one Gradio chatbot message using the complete RAG pipeline.
    """

    message = message.strip()

    if not message:
        return "Please enter a question."

    try:
        answer, sources = answer_question(
            question=message,
            history=history,
            top_k=8,
            faiss_k=50,
            bm25_k=20,
        )

        sources_markdown = format_sources(sources)

        return f"{answer}\n\n---\n\n{sources_markdown}"

    except Exception as error:
        return (
            "I could not process this question because an error occurred.\n\n"
            f"`{type(error).__name__}: {error}`"
        )

In [ ]:
test_response = research_chat(
    message="Which GitHub projects involve generative AI?",
    history=[],
)

display(Markdown(test_response))

In [ ]:
with gr.Blocks(
    title="ResearchGPT — Milad Saeedi",
) as demo:

    gr.Markdown(
        """
        # ResearchGPT — Milad Saeedi

        Explore Milad Saeedi’s research, PhD thesis, publications,
        machine learning experience, and GitHub projects.

        Answers are generated from retrieved portfolio documents and
        include supporting sources.
        """
    )

    chatbot = gr.Chatbot(
        placeholder=(
            "Ask about Milad's research, publications, "
            "technical skills, or GitHub projects."
        ),
        height=550,
    )

    gr.ChatInterface(
        fn=research_chat,
        chatbot=chatbot,
        examples=[
            "What makes Milad a good ML engineer?",
            "Which GitHub projects demonstrate deep learning?",
            "How has Milad used computer vision?",
            "Summarize his PhD research.",
            "Which repositories use PyTorch?",
            "What experience does Milad have with LLMs?",
            "Summarize Milad Saeedi's research.",
            "What are the main contributions of his PhD thesis?",
            "Summarize his GitHub GAN projects.",
            "Which projects involve computer vision?",
            "What experience does he have with LoRA and GRPO?",
            "What did his research show about spatial cross-validation?",
            "Tell me about his geospatial modeling experience.",
            "How can I contact Milad?",
            
            ],
        save_history=True,
        flagging_mode="never",
    )

    gr.Markdown(
        """
        ---
        **ResearchGPT** uses hybrid retrieval, BGE embeddings,
        FAISS, BM25, cross-encoder reranking, and grounded answer generation.
        """
    )

In [ ]:
demo.queue().launch()